# 04c – DistilBERT fine-tuning (GPU: Colab or local NVIDIA)
Pre-trained Transformer fine-tuned on our tickets. Run all → upload `data/processed/*.csv` if asked → unzip `distilbert_outputs.zip` into `results/distilbert/` → run `04c_distilbert_results.ipynb`.

In [ ]:
import os, sys, json, time, random, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score

SMOKE = os.environ.get("SMOKE") == "1"   # tiny CPU run used only to test this notebook locally
IN_COLAB = "google.colab" in sys.modules
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | transformers {transformers.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))
if IN_COLAB and device.type != "cuda":
    raise SystemExit("No GPU found: Runtime → Change runtime type → T4 GPU, then Run all again.")

## 1. Load splits

In [ ]:
LABELS = ["Hardware", "HR Support", "Access", "Miscellaneous", "Storage", "Purchase", "Internal Project", "Administrative rights"]
SEED = 42

local_dir = Path("../data/processed")
if (local_dir / "train.csv").exists():
    data_dir = local_dir
else:
    data_dir = Path(".")
    missing = [f for f in ["train.csv", "val.csv", "test.csv"] if not (data_dir / f).exists()]
    if missing:
        from google.colab import files
        print("Upload:", missing)
        files.upload()

train, val, test = (pd.read_csv(data_dir / f"{n}.csv") for n in ("train", "val", "test"))
if SMOKE:
    train, val, test = train.sample(256, random_state=SEED), val.sample(128, random_state=SEED), test.sample(128, random_state=SEED)
label_to_id = {l: i for i, l in enumerate(LABELS)}
for d in (train, val, test):
    d["y"] = d.label.map(label_to_id)
print({n: len(d) for n, d in [("train", train), ("val", val), ("test", test)]})

## 2. Tokenize

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 64 if SMOKE else 256
EPOCHS = 1 if SMOKE else 4
BATCH = 32
LR = 3e-5

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
lens = [len(tokenizer(t)["input_ids"]) for t in train.text.sample(min(3000, len(train)), random_state=SEED)]
print(f"Tokens per ticket: median {np.median(lens):.0f}, 95th pct {np.percentile(lens, 95):.0f}; "
      f"covered by {MAX_LEN}: {np.mean(np.array(lens) <= MAX_LEN):.1%}")
print("Example:", tokenizer.tokenize(train.text.iloc[0])[:20])

def encode(df):
    enc = tokenizer(list(df.text), truncation=True, max_length=MAX_LEN)
    return [{"input_ids": enc["input_ids"][i], "attention_mask": enc["attention_mask"][i], "labels": int(y)}
            for i, y in enumerate(df.y)]

collate = DataCollatorWithPadding(tokenizer)   # pads each batch only to its longest ticket
g = torch.Generator().manual_seed(SEED)
train_dl = DataLoader(encode(train), batch_size=BATCH, shuffle=True, collate_fn=collate, generator=g)
val_dl = DataLoader(encode(val), batch_size=128, collate_fn=collate)
test_dl = DataLoader(encode(test), batch_size=128, collate_fn=collate)

## 3. Model, optimiser, weighted loss

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=label_to_id).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

counts = np.bincount(train.y, minlength=len(LABELS))
class_weights = torch.tensor(len(train) / (len(LABELS) * counts), dtype=torch.float, device=device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

no_decay = ["bias", "LayerNorm.weight"]
param_groups = [
    {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": 0.01},
    {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
]
optimizer = torch.optim.AdamW(param_groups, lr=LR)
total_steps = EPOCHS * len(train_dl)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

## 4. Fine-tune
Best epoch chosen on validation macro-F1.

In [ ]:
@torch.no_grad()
def predict(dl):
    model.eval()
    probs, losses, n = [], 0.0, 0
    for batch in dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
        losses += loss_fn(logits.float(), batch["labels"]).item() * len(batch["labels"])
        n += len(batch["labels"])
        probs.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.concatenate(probs), losses / n

history = []
best_f1, best_state = -1, None
out_dir = Path("distilbert_outputs"); out_dir.mkdir(exist_ok=True)
t_start = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    run_loss, run_correct, run_n = 0.0, 0, 0
    t0 = time.time()
    for step, batch in enumerate(train_dl, 1):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
            loss = loss_fn(logits.float(), batch["labels"])
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        run_loss += loss.item() * len(batch["labels"])
        run_correct += (logits.argmax(-1) == batch["labels"]).sum().item()
        run_n += len(batch["labels"])
        if step % 200 == 0:
            print(f"  epoch {epoch} step {step}/{len(train_dl)} loss {run_loss / run_n:.4f}", flush=True)
    val_proba, val_loss = predict(val_dl)
    val_pred = val_proba.argmax(1)
    rec = {"epoch": epoch, "loss": run_loss / run_n, "accuracy": run_correct / run_n, "val_loss": val_loss,
           "val_accuracy": float((val_pred == val.y.values).mean()),
           "val_macro_f1": float(f1_score(val.y, val_pred, average="macro")), "epoch_seconds": time.time() - t0}
    history.append(rec)
    print(f"epoch {epoch}: train loss {rec['loss']:.4f} acc {rec['accuracy']:.4f} | val loss {val_loss:.4f} "
          f"acc {rec['val_accuracy']:.4f} macro-F1 {rec['val_macro_f1']:.4f} | {rec['epoch_seconds'] / 60:.1f} min", flush=True)
    if rec["val_macro_f1"] > best_f1:
        best_f1, best_epoch = rec["val_macro_f1"], epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
train_seconds = time.time() - t_start
model.load_state_dict(best_state)
print(f"Best epoch: {best_epoch} (val macro-F1 {best_f1:.4f}) | total training {train_seconds / 60:.1f} min")

## 5. Predict and export

In [ ]:
def export_preds(df, proba, name):
    preds = pd.DataFrame(proba, columns=[f"p_{l}" for l in LABELS])
    preds.insert(0, "text", df.text.values)
    preds.insert(1, "label", df.label.values)
    preds.insert(2, "pred", np.array(LABELS)[proba.argmax(1)])
    preds.insert(3, "confidence", proba.max(1))
    preds.to_csv(out_dir / f"preds_{name}.csv", index=False)
    return preds

val_proba, _ = predict(val_dl)
test_proba, _ = predict(test_dl)
export_preds(val, val_proba, "val")
test_preds = export_preds(test, test_proba, "test")
print(f"Test accuracy {(test_preds.pred == test_preds.label).mean():.4f} | "
      f"macro-F1 {f1_score(test_preds.label, test_preds.pred, average='macro'):.4f}")

run_info = {"model_name": MODEL_NAME, "max_len": MAX_LEN, "epochs": EPOCHS, "batch_size": BATCH, "lr": LR,
            "warmup": 0.1, "weight_decay": 0.01, "best_epoch": best_epoch, "trainable_params": n_params,
            "train_seconds": train_seconds, "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
            "torch": torch.__version__, "transformers": transformers.__version__, "smoke_test": SMOKE, "history": history}
(out_dir / "run_info.json").write_text(json.dumps(run_info, indent=2))
shutil.make_archive("distilbert_outputs", "zip", out_dir)
print("Wrote distilbert_outputs.zip:", sorted(p.name for p in out_dir.iterdir()))
if IN_COLAB:
    from google.colab import files
    files.download("distilbert_outputs.zip")

## 6. (Optional) Download the fine-tuned model
Set `DOWNLOAD_MODEL = True` for the demo app (~250 MB).

In [ ]:
DOWNLOAD_MODEL = False
if DOWNLOAD_MODEL:
    model.save_pretrained("distilbert_model")
    tokenizer.save_pretrained("distilbert_model")
    shutil.make_archive("distilbert_model", "zip", "distilbert_model")
    if IN_COLAB:
        from google.colab import files
        files.download("distilbert_model.zip")